In [2]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to query the warehouse.")

Connected. Ready to query the warehouse.


In [4]:
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face token: ")

Paste your Hugging Face token:  ········


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gopinath04-R/gopinath-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content page, on one report date, for one client (`content_hash_id` + `client_hash_id` + `report_date`).

**Time window:** I'll build against one middle month first — `month=2026-03` from `fact_content_daily_performance` — to avoid Hugging Face rate limits while developing.


In [6]:
q = f"""
SELECT COUNT(*) AS row_count, COUNT(DISTINCT content_hash_id) AS unique_pages, COUNT(DISTINCT client_hash_id) AS unique_clients
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q)

┌───────────┬──────────────┬────────────────┐
│ row_count │ unique_pages │ unique_clients │
│   int64   │    int64     │     int64      │
├───────────┼──────────────┼────────────────┤
│   9841378 │       331437 │             55 │
└───────────┴──────────────┴────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** impressions_90d, clicks, avg_position, ctr, word_count, content_age_days, days_since_last_update — all observed before the decision point.

**Label/proxy:** trend_direction == "down" (declining flag).

**Context (not modeled):** client_hash_id, content_hash_id, report_date — used for joins/grouping only.

**Excluded:** any FlyRank product decision flags (health_score, priority_score, action_type) — excluded because using them would just teach the model to copy an existing rule instead of finding real signal (circular result / leakage).

In [7]:
q2 = f"""
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 3
"""
con.sql(q2)

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verifying the contract claims above with real queries: row grain, missing values, and window boundaries.

In [9]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell you:** It's an unbalanced panel — different clients have different history depths, so comparing an old client to a new one isn't apples-to-apples. Rows before a client's GA4 start have search data only (ga4_data_available = FALSE), so "no tracking yet" must never be read as "no traffic." It also can't prove causation — a page ranked as "needs refresh" is a review priority, not a guarantee that fixing it will recover traffic.

In [10]:
q4 = f"""
SELECT ga4_data_available, COUNT(*) AS rows
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY ga4_data_available
"""
con.sql(q4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬─────────┐
│ ga4_data_available │  rows   │
│      boolean       │  int64  │
├────────────────────┼─────────┤
│ false              │ 6408671 │
│ NULL               │ 3018741 │
│ true               │  413966 │
└────────────────────┴─────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.